# start here: https://www.rightmove.co.uk/property-for-sale/find.html?useLocationIdentifier=true&locationIdentifier=REGION%5E162&radius=0.0&_includeSSTC=on&sortType=6&channel=BUY&transactionType=BUY&displayLocationIdentifier=Birmingham.html

In [ ]:
import re
import time
import json
import math
from typing import List, Dict, Optional
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd

START_URL = ("https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&radius=40.0"
             "&locationIdentifier=USERDEFINEDAREA%5E%7B%22polylines%22%3A%22ctl%60Ijy%7DLjI%7DgkC%3Fm%7Bp%40rlgBc%7B%40kYlrcEs%7CfB%7DpC%22%7D"
             "&transactionType=BUY&displayLocationIdentifier=undefined&maxDaysSinceAdded=14"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
    "Connection": "keep-alive",
    "DNT": "1",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

PROPERTY_LINK_SELECTORS = [
    "a.propertyCard-link",            # common
    "a.propertyCard-detailsLink",     # alternative
    "a[data-test='property-card-link']",  # sometimes used
]

def get_soup(url: str) -> Optional[BeautifulSoup]:
    try:
        resp = SESSION.get(url, timeout=30)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "html.parser")
    except requests.RequestException:
        return None

def extract_property_links(listing_url: str) -> List[str]:
    soup = get_soup(listing_url)
    if soup is None:
        return []
    links: List[str] = []
    for selector in PROPERTY_LINK_SELECTORS:
        for a in soup.select(selector):
            href = a.get("href")
            if not href:
                continue
            full = urljoin("https://www.rightmove.co.uk/", href)
            # ensure it's a property URL
            if "/properties/" in full:
                links.append(full.split("#")[0])
    # dedupe
    return sorted(set(links))

PROPERTY_JSON_PATTERNS = [
    re.compile(r"window\.__PRELOADED_STATE__\s*=\s*(\{.*?\})\s*;", re.S),
    re.compile(r"__NEXT_DATA__\"\s*:\s*(\{.*?\})\s*[,<]", re.S),
    re.compile(r"window\.__INITIAL_STATE__\s*=\s*(\{.*?\})\s*;", re.S),
    re.compile(r"window\.INITIAL_STATE\s*=\s*(\{.*?\})\s*;", re.S),
    re.compile(r"window\.__RM_STATE__\s*=\s*(\{.*?\})\s*;", re.S),
]

INT_PATTERN = re.compile(r"(\d+)")
SQM_PATTERN = re.compile(r"([0-9,.]+)\s*(sq\s*m|sqm|square\s*metres?)", re.I)
SQFT_PATTERN = re.compile(r"([0-9,.]+)\s*(sq\s*ft|sqft|square\s*feet)\b", re.I)

def first_int(text: str) -> Optional[int]:
    if not text:
        return None
    m = INT_PATTERN.search(text)
    return int(m.group(1)) if m else None

def find_text_case_insensitive(soup: BeautifulSoup, label: str) -> Optional[str]:
    if soup is None:
        return None
    # look for label followed by value in list items or definition lists
    # common Rightmove markup varies; keep flexible
    for li in soup.select("li"):
        t = li.get_text(" ", strip=True)
        if label.lower() in t.lower():
            return t
    for dt in soup.select("dt"):
        if label.lower() in dt.get_text(strip=True).lower():
            dd = dt.find_next("dd")
            if dd:
                return dd.get_text(" ", strip=True)
    return None

def parse_features_list(soup: BeautifulSoup) -> List[str]:
    features: List[str] = []
    # include common ULs and any ULs within the main article body
    for ul_sel in [
        "ul.key-features",
        "ul.property-features",
        "ul[data-test='key-features']",
        "article ul",
    ]:
        for ul in soup.select(ul_sel):
            for li in ul.select("li"):
                text = li.get_text(" ", strip=True)
                if text:
                    features.append(text)
    # fallback: look for section heading
    for heading in soup.find_all(["h2", "h3" ]):
        if "key feature" in heading.get_text(strip=True).lower():
            ul = heading.find_next("ul")
            if ul:
                for li in ul.select("li"):
                    text = li.get_text(" ", strip=True)
                    if text:
                        features.append(text)
    # dedupe while preserving order
    seen = set(); deduped = []
    for f in features:
        if f not in seen:
            seen.add(f); deduped.append(f)
    return deduped

def try_parse_embedded_json(html: str) -> List[Dict]:
    """Return a list of embedded JSON candidates (app state and JSON-LD blocks)."""
    candidates: List[Dict] = []
    # App/state blobs
    for pat in PROPERTY_JSON_PATTERNS:
        m = pat.search(html)
        if m:
            text = m.group(1)
            for payload in (text, text.replace("\n", " ").replace("\r", " ")):
                try:
                    obj = json.loads(payload)
                    if isinstance(obj, dict):
                        candidates.append(obj)
                        break
                except Exception:
                    continue
    # JSON-LD blocks
    try:
        soup = BeautifulSoup(html, "html.parser")
        for script in soup.select('script[type="application/ld+json"]'):
            try:
                data = json.loads(script.get_text(strip=True))
                if isinstance(data, dict):
                    candidates.append(data)
                elif isinstance(data, list):
                    for item in data:
                        if isinstance(item, dict):
                            candidates.append(item)
            except Exception:
                continue
    except Exception:
        pass
    return candidates

def extract_detail(url: str) -> Dict:
    result: Dict = {
        "url": url,
        "property_id": None,
        "address": None,
        "price": None,
        "property_type": None,
        "bedrooms": None,
        "bathrooms": None,
        "size_sq_m": None,
        "size_sq_ft": None,
        "tenure": None,
        "council_tax_band": None,
        "parking": None,
        "garden": None,
        "key_features": None,
    }

    try:
        resp = SESSION.get(url, timeout=30)
        resp.raise_for_status()
    except requests.RequestException:
        return result

    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    # property id
    try:
        pid = re.search(r"/properties/(\d+)", url)
        result["property_id"] = pid.group(1) if pid else None
    except Exception:
        pass

    # price (listing detail pages often have data-testid on price)
    price_sel = soup.select_one("[itemtype='https://schema.org/Residence'] div article div div div > span:nth-of-type(1)") \
                or soup.select_one("[data-testid='price']") \
                or soup.select_one(".property-header-price")
    if price_sel:
        result["price"] = price_sel.get_text(" ", strip=True)

    # address
    addr_sel = soup.select_one("[data-testid='address']") \
            or soup.select_one(".property-header-bedroom-and-price .text") \
            or soup.select_one(".address, [itemprop='address']")
    if addr_sel:
        result["address"] = addr_sel.get_text(" ", strip=True)

    # Title-based fallbacks (often includes "5 bedroom detached house for sale" etc.)
    title_el = soup.select_one("[data-testid='title']") or soup.select_one("h1")
    if title_el:
        title_text = title_el.get_text(" ", strip=True)
        if result["bedrooms"] is None:
            m = re.search(r"(\d+)\s*bed(room)?", title_text, re.I)
            if m:
                result["bedrooms"] = int(m.group(1))
        if result["property_type"] is None:
            m = re.search(r"\b(Detached|Semi[- ]?Detached|End of Terrace|Terraced|Flat|Apartment|Bungalow|Cottage|Townhouse)\b", title_text, re.I)
            if m:
                result["property_type"] = m.group(1).replace("-", " ").title()

    # Fact chips (rounded facts)
    facts = [el.get_text(" ", strip=True) for el in soup.select("[data-testid='rounded-fact'], .key-fact, .fact")]
    for fact in facts:
        if result["bedrooms"] is None:
            m = re.search(r"(\d+)\s*bed", fact, re.I)
            if m:
                result["bedrooms"] = int(m.group(1))
        if result["bathrooms"] is None:
            m = re.search(r"(\d+)\s*bath", fact, re.I)
            if m:
                result["bathrooms"] = int(m.group(1))
        if result["size_sq_ft"] is None:
            m = SQFT_PATTERN.search(fact)
            if m:
                result["size_sq_ft"] = m.group(1).replace(",", "")
        if result["size_sq_m"] is None:
            m = SQM_PATTERN.search(fact)
            if m:
                result["size_sq_m"] = m.group(1).replace(",", "")

    # Meta tag fallbacks
    if result["address"] is None:
        meta = soup.select_one('meta[property="og:title"]') or soup.select_one('meta[name="twitter:title"]')
        if meta and meta.get("content"):
            c = meta["content"].strip()
            # Often formatted like "5 bed detached house for sale in ADDRESS - Rightmove"
            part = c.split(" - ")[0]
            # If it contains a comma, likely address after the type phrase
            if "," in part:
                result["address"] = part.split(" in ")[-1].strip()
    if result["price"] is None:
        # gentle text scrape for a pound amount
        m = re.search(r"£\s*[\d,]+", soup.get_text(" ", strip=True))
        if m:
            result["price"] = m.group(0)

    # key features
    features = parse_features_list(soup)
    if features:
        result["key_features"] = "; ".join(features)
        lf = " ".join(f.lower() for f in features)
        if any("parking" in f for f in lf.split("; ")) or "driveway" in lf:
            result["parking"] = "Yes"
        if "garden" in lf or "balcony" in lf:
            result["garden"] = "Yes"

    # tenure / council tax from text blocks
    tenure_text = find_text_case_insensitive(soup, "Tenure")
    if tenure_text:
        result["tenure"] = tenure_text
    ct_text = find_text_case_insensitive(soup, "Council Tax")
    if ct_text:
        # often like "Council Tax Band: D"
        m = re.search(r"Band\s*:?\s*([A-H])", ct_text, re.I)
        result["council_tax_band"] = m.group(1).upper() if m else ct_text

    # try embedded JSON for structured fields
    data_candidates = try_parse_embedded_json(html)
    for data in data_candidates:
        # Rightmove shapes vary; try common paths
        try:
            # JSON-LD schema.org
            if data.get("@type") in ("House", "Apartment", "SingleFamilyResidence", "Residence"):
                agg_offer = data.get("offers") or data.get("aggregateRating")
                if isinstance(agg_offer, dict):
                    price = agg_offer.get("price") or agg_offer.get("lowPrice") or agg_offer.get("highPrice")
                    if price and result["price"] is None:
                        result["price"] = str(price)
                addr = data.get("address")
                if isinstance(addr, dict) and result["address"] is None:
                    parts = [addr.get(k) for k in ["streetAddress", "addressLocality", "addressRegion", "postalCode"] if addr.get(k)]
                    if parts:
                        result["address"] = ", ".join(parts)
                if result["bedrooms"] is None and isinstance(data.get("numberOfRooms"), int):
                    result["bedrooms"] = data.get("numberOfRooms")
                # floor size
                fs = data.get("floorSize")
                if isinstance(fs, dict):
                    val = fs.get("value") or fs.get("area")
                    unit = (fs.get("unitCode") or fs.get("unitText") or "").lower()
                    if val:
                        if "m" in unit and result["size_sq_m"] is None:
                            result["size_sq_m"] = str(val)
                        if "ft" in unit and result["size_sq_ft"] is None:
                            result["size_sq_ft"] = str(val)

            # Find property node within app state
            node = None
            if "props" in data:
                node = (
                    data.get("props", {}).get("pageProps", {}).get("propertyData")
                    or data.get("props", {}).get("pageProps", {}).get("listing")
                )
            if node is None and "details" in data:
                node = data.get("details")
            if node is None:
                def _find_property(d):
                    if isinstance(d, dict):
                        if any(k in d for k in ["bedrooms", "bathrooms", "propertySubType", "displaySize", "keyFeatures"]):
                            return d
                        for v in d.values():
                            f = _find_property(v)
                            if f is not None:
                                return f
                    elif isinstance(d, list):
                        for v in d:
                            f = _find_property(v)
                            if f is not None:
                                return f
                    return None
                node = _find_property(data)

            if node:
                if result["bedrooms"] is None:
                    val = node.get("bedrooms") or node.get("bedroomsCount")
                    result["bedrooms"] = val
                if result["bathrooms"] is None:
                    val = node.get("bathrooms") or node.get("bathroomsCount")
                    result["bathrooms"] = val
                ptype = node.get("propertySubType") or node.get("propertyType")
                if ptype:
                    result["property_type"] = str(ptype)
                disp_size = node.get("displaySize") or node.get("size")
                if isinstance(disp_size, str):
                    m2 = SQM_PATTERN.search(disp_size)
                    ft2 = SQFT_PATTERN.search(disp_size)
                    if m2 and result["size_sq_m"] is None:
                        result["size_sq_m"] = m2.group(1).replace(",", "")
                    if ft2 and result["size_sq_ft"] is None:
                        result["size_sq_ft"] = ft2.group(1).replace(",", "")
                if result["tenure"] is None:
                    ten = node.get("tenure") or node.get("tenureType")
                    if ten:
                        result["tenure"] = str(ten)
                if result["council_tax_band"] is None:
                    tax = node.get("councilTax") or node.get("councilTaxBand")
                    if isinstance(tax, dict):
                        band = tax.get("band") or tax.get("value")
                        if band:
                            result["council_tax_band"] = str(band)
                    elif tax:
                        result["council_tax_band"] = str(tax)
                if result["address"] is None:
                    addr = node.get("address") or node.get("displayAddress")
                    if isinstance(addr, dict):
                        result["address"] = addr.get("displayAddress") or addr.get("addressLine1")
                    elif addr:
                        result["address"] = str(addr)
                if result["price"] is None:
                    price = node.get("price") or node.get("displayPrice")
                    if isinstance(price, dict):
                        result["price"] = price.get("displayPrices", [{}])[0].get("displayPrice") or price.get("amount")
                    elif price:
                        result["price"] = str(price)
                if result["key_features"] is None:
                    feats = node.get("keyFeatures") or node.get("features")
                    if isinstance(feats, list):
                        cleaned = [str(x).strip() for x in feats if str(x).strip()]
                        if cleaned:
                            result["key_features"] = "; ".join(cleaned)
        except Exception:
            continue

    # dt/dd selectors from your sitemap for type/beds/baths/size
    def dd_text(label: str) -> Optional[str]:
        # BeautifulSoup doesn't support :contains in CSS; scan dt elements manually
        for dt in soup.select("dt"):
            if label.lower() in dt.get_text(strip=True).lower():
                dd = dt.find_next_sibling("dd")
                if dd:
                    p = dd.find("p")
                    return (p.get_text(" ", strip=True) if p else dd.get_text(" ", strip=True)) or None
        return None

    if result["property_type"] is None:
        t = dd_text("PROPERTY TYPE")
        if t:
            result["property_type"] = t
    if result["bedrooms"] is None:
        b = dd_text("BEDROOMS"); result["bedrooms"] = first_int(b) if b else None
    if result["bathrooms"] is None:
        bth = dd_text("BATHROOMS"); result["bathrooms"] = first_int(bth) if bth else None
    if result["size_sq_m"] is None or result["size_sq_ft"] is None:
        size_text = dd_text("SIZE")
        if size_text:
            m2 = SQM_PATTERN.search(size_text); ft2 = SQFT_PATTERN.search(size_text)
            if m2 and result["size_sq_m"] is None:
                result["size_sq_m"] = m2.group(1).replace(",", "")
            if ft2 and result["size_sq_ft"] is None:
                result["size_sq_ft"] = ft2.group(1).replace(",", "")

    # council tax, parking, garden via dt/dd if present
    if result["council_tax_band"] is None:
        ct = dd_text("COUNCIL TAX")
        if ct:
            m = re.search(r"Band\s*:?\s*([A-H])", ct, re.I)
            result["council_tax_band"] = m.group(1).upper() if m else ct
    if result["parking"] is None:
        pk = dd_text("PARKING")
        if pk:
            result["parking"] = "Yes" if re.search(r"yes|private|drive|garage|off[- ]road|parking", pk, re.I) else pk
    if result["garden"] is None:
        gd = dd_text("GARDEN")
        if gd:
            result["garden"] = "Yes" if re.search(r"yes|garden|yard|balcony|terrace", gd, re.I) else gd

    # fallback bedrooms/bathrooms/type from visible summary header
    if result["bedrooms"] is None or result["bathrooms"] is None or result["property_type"] is None:
        summary = soup.select_one("[data-testid='title']") or soup.select_one(".property-header-bedroom-and-price")
        if summary:
            text = summary.get_text(" ", strip=True)
            if result["bedrooms"] is None:
                result["bedrooms"] = first_int(re.search(r"(\d+)\s*bed", text, re.I).group(0)) if re.search(r"(\d+)\s*bed", text, re.I) else None
            if result["bathrooms"] is None:
                result["bathrooms"] = first_int(re.search(r"(\d+)\s*bath", text, re.I).group(0)) if re.search(r"(\d+)\s*bath", text, re.I) else None
            if result["property_type"] is None:
                m = re.search(r"\b(Detached|Semi-Detached|Terraced|End of Terrace|Flat|Apartment|Bungalow|Cottage|Townhouse)\b", text, re.I)
                result["property_type"] = m.group(1).title() if m else None

    return result


def paginate_and_scrape(start_url: str, max_pages: int = 5, delay_s: float = 1.5) -> pd.DataFrame:
    # Rightmove paginates by index in steps of 24
    all_rows: List[Dict] = []
    parsed = urlparse(start_url)

    def with_index(idx: int) -> str:
        if "index=" in start_url:
            return re.sub(r"index=\d+", f"index={idx}", start_url)
        sep = "&" if parsed.query else "?"
        return f"{start_url}{sep}index={idx}"

    for page_no in range(max_pages):
        idx = page_no * 24
        url = with_index(idx)
        print(f"Listing page {page_no+1}/{max_pages}: index={idx}")
        links = extract_property_links(url)
        if not links:
            print("No property links found, stopping.")
            break
        print(f"Found {len(links)} property links")
        for i, link in enumerate(links, 1):
            print(f"  [{i}/{len(links)}] {link}")
            try:
                row = extract_detail(link)
                all_rows.append(row)
            except Exception as e:
                print(f"    Error: {e}")
            time.sleep(delay_s)
        # polite pause between listing pages
        time.sleep(2.0)

    df = pd.DataFrame(all_rows)
    return df

# Run multi-county scrape
MAX_PAGES_TO_SCRAPE = 100000  # adjust if needed

county_urls = [
    (
        "West Midlands",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61378&transactionType=BUY&displayLocationIdentifier=West-Midlands-County.html",
    ),
    (
        "Warwickshire",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61327&transactionType=BUY&displayLocationIdentifier=Warwickshire.html",
    ),
    (
        "Worcestershire",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61335&transactionType=BUY&displayLocationIdentifier=Worcestershire.html",
    ),
    (
        "Shropshire",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61325&transactionType=BUY&displayLocationIdentifier=Shropshire.html",
    ),
    (
        "Staffordshire",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61326&transactionType=BUY&displayLocationIdentifier=Staffordshire.html",
    ),
    (
        "Herefordshire",
        "https://www.rightmove.co.uk/property-for-sale/find.html?sortType=2&areaSizeUnit=sqft&viewType=LIST&channel=BUY&index=0&locationIdentifier=REGION%5E61315&transactionType=BUY&displayLocationIdentifier=Herefordshire.html",
    ),
]

import re

def extract_outward_postcode(address: str):
    if not isinstance(address, str) or not address.strip():
        return None
    text = address.upper().strip().rstrip(",.;")
    m = re.search(r"([A-Z]{1,2}\d{1,2}[A-Z]?)\s*\d[ABD-HJLNP-UW-Z]{2}$", text)
    return m.group(1) if m else None

all_frames = []
for county_name, url in county_urls:
    print(f"\n=== Scraping {county_name} ===")
    df = paginate_and_scrape(url, max_pages=MAX_PAGES_TO_SCRAPE)
    if df.empty:
        continue
    df["County"] = county_name
    df["district"] = df["address"].apply(extract_outward_postcode)
    all_frames.append(df)

if all_frames:
    results_df = pd.concat(all_frames, ignore_index=True)
    # Insert County after property_id
    cols = list(results_df.columns)
    if "County" in cols and "property_id" in cols:
        cols.remove("County")
        insert_at = cols.index("property_id") + 1
        cols.insert(insert_at, "County")
        results_df = results_df[cols]
    print(f"Collected {len(results_df)} properties across {len(all_frames)} counties")
    output_path = "rightmove_west_midlands_properties.csv"
    results_df.to_csv(output_path, index=False)
    print(f"Saved -> {output_path}")
else:
    print("No data scraped.")



In [ ]:
# Outcode enrichment and outcode-based scrape (test: B93, B92, B94)
import pandas as pd
import re
from urllib.parse import urlencode

# 1) Read outward postcodes and enrich with population and square miles
outcodes_path = "outward_postcodes.csv"
oc_df = pd.read_csv(outcodes_path)

# Detect outcode column
col_candidates = ["OutwardPostcode", "Outward Postcode", "outward_postcode", "outcode", "district"]
outcode_col = None
for c in col_candidates:
    if c in oc_df.columns:
        outcode_col = c
        break
if outcode_col is None:
    raise ValueError("Could not find an outcode column. Expected one of: " + ", ".join(col_candidates))

# Normalize
oc_df["outcode"] = (
    oc_df[outcode_col]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)
oc_df = oc_df[["outcode"]].drop_duplicates()

# Load mapping for population and area
map_df = pd.read_csv("outcodes_final_table.csv")
map_df["outcode"] = map_df["district"].astype(str).str.upper().str.replace(r"\s+", "", regex=True)

enriched = oc_df.merge(
    map_df[["outcode", "population", "area_sqmi_rounded"]], on="outcode", how="left"
).rename(columns={"area_sqmi_rounded": "area_sq_mi"})

enriched.to_csv("outward_postcodes_pop_sqm.csv", index=False)
print(f"Saved enrichment -> outward_postcodes_pop_sqm.csv ({len(enriched)} rows)")

# 2) Helper: outward postcode from address

def extract_outward_postcode(address: str):
    if not isinstance(address, str) or not address.strip():
        return None
    text = address.upper().strip().rstrip(",.;")
    m = re.search(r"([A-Z]{1,2}\d{1,2}[A-Z]?)\s*\d[ABD-HJLNP-UW-Z]{2}$", text)
    return m.group(1) if m else None

# 3) Build Rightmove URL for an outcode

def build_outcode_start_url(outcode: str) -> str:
    params = {
        "searchLocation": outcode,
        "useLocationIdentifier": "true",
        # Use explicit OUTCODE^<outcode>
        "locationIdentifier": f"OUTCODE^{outcode}",
        "buy": "For sale",
        "radius": "0.0",
        "_includeSSTC": "on",
        "sortType": "2",
        "channel": "BUY",
        "transactionType": "BUY",
        "index": "0",
    }
    # Keep caret unencoded for Rightmove
    return "https://www.rightmove.co.uk/property-for-sale/find.html?" + urlencode(params, safe="^")

# 4) Test only these outcodes first
TEST_OUTCODES = ["B93", "B92", "B94"]
MAX_PAGES_TO_SCRAPE = 5

all_frames = []
for oc in TEST_OUTCODES:
    start_url = build_outcode_start_url(oc)
    print(f"\n=== Scraping OUTCODE {oc} ===")
    df = paginate_and_scrape(start_url, max_pages=MAX_PAGES_TO_SCRAPE)
    if df.empty:
        continue
    df["outcode"] = oc
    df["district"] = df["address"].apply(extract_outward_postcode)
    all_frames.append(df)

if all_frames:
    results_df = pd.concat(all_frames, ignore_index=True)
    # Insert outcode after property_id
    cols = list(results_df.columns)
    if "outcode" in cols and "property_id" in cols:
        cols.remove("outcode")
        insert_at = cols.index("property_id") + 1
        cols.insert(insert_at, "outcode")
        results_df = results_df[cols]
    print(f"Collected {len(results_df)} properties across {len(all_frames)} outcodes")
    out_path = "rightmove_outcodes_test.csv"
    results_df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")
else:
    print("No data scraped for test outcodes.")



In [ ]:
# Enrich outcodes and test outcode-based scraping (B93, B92, B94)
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlencode, urlparse, parse_qs

# --- Fallback: define scraper helpers if missing (fresh kernel) ---
if "SESSION" not in globals():
    HEADERS = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-GB,en;q=0.9",
        "Connection": "keep-alive",
        "DNT": "1",
    }
    SESSION = requests.Session()
    SESSION.headers.update(HEADERS)

PROPERTY_LINK_SELECTORS = [
    "a.propertyCard-link",
    "a.propertyCard-detailsLink",
    "a[data-test='property-card-link']",
]

def get_soup(url: str):
    try:
        resp = SESSION.get(url, timeout=30)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "html.parser")
    except requests.RequestException:
        return None

def extract_property_links(listing_url: str):
    soup = get_soup(listing_url)
    if soup is None:
        return []
    links = []
    for selector in PROPERTY_LINK_SELECTORS:
        for a in soup.select(selector):
            href = a.get("href")
            if not href:
                continue
            full = "https://www.rightmove.co.uk" + href if href.startswith("/") else href
            if "/properties/" in full:
                links.append(full.split("#")[0])
    return sorted(set(links))

# 1) Read outward postcodes and enrich with population and square miles
outcodes_path = "outward_postcodes.csv"
try:
    oc_df = pd.read_csv(outcodes_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Missing {outcodes_path}. Create it with a column of outcodes, e.g. 'OutwardPostcode'.")

# Normalize column name
col_candidates = ["OutwardPostcode", "Outward Postcode", "outward_postcode", "outcode", "district"]
outcode_col = None
for c in col_candidates:
    if c in oc_df.columns:
        outcode_col = c
        break
if outcode_col is None:
    raise ValueError("Could not find an outcode column. Expected one of: " + ", ".join(col_candidates))

oc_df["outcode"] = (
    oc_df[outcode_col]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)
oc_df = oc_df[["outcode"]].drop_duplicates()

# Load local mapping of outcode -> population, area (sq mi)
map_df = pd.read_csv("outcodes_final_table.csv")
map_df["outcode"] = (
    map_df["district"].astype(str).str.upper().str.replace(r"\s+", "", regex=True)
)

enriched = oc_df.merge(
    map_df[["outcode", "population", "area_sqmi_rounded"]], on="outcode", how="left"
).rename(columns={"area_sqmi_rounded": "area_sq_mi"})

enriched.to_csv("outward_postcodes_pop_sqm.csv", index=False)
print(f"Saved enrichment -> outward_postcodes_pop_sqm.csv ({len(enriched)} rows)")

# 2) Helper to extract outward postcode from address

def extract_outward_postcode(address: str):
    if not isinstance(address, str) or not address.strip():
        return None
    text = address.upper().strip().rstrip(",.;")
    m = re.search(r"([A-Z]{1,2}\d{1,2}[A-Z]?)\s*\d[ABD-HJLNP-UW-Z]{2}$", text)
    return m.group(1) if m else None

# 3) Resolve numeric OUTCODE identifier; build URLs and fallback to JSON API when needed

OUTCODE_ID_STATIC = {
    "B93": "OUTCODE^116",
    # Add more known mappings here as you discover them:
    # "B92": "OUTCODE^<id>",
    # "B94": "OUTCODE^<id>",
}
OUTCODE_ID_CACHE = {}


def _search_json_for_outcode_identifier(obj: object) -> str | None:
    # Recursively scan JSON for strings like 'OUTCODE^1234'
    if isinstance(obj, dict):
        for v in obj.values():
            found = _search_json_for_outcode_identifier(v)
            if found:
                return found
    elif isinstance(obj, list):
        for v in obj:
            found = _search_json_for_outcode_identifier(v)
            if found:
                return found
    elif isinstance(obj, str):
        m = re.search(r"OUTCODE\^(\d+)", obj)
        if m:
            return f"OUTCODE^{m.group(1)}"
    return None


def resolve_outcode_identifier(outcode: str) -> str | None:
    outcode = outcode.upper().strip()
    # 0) Hardcoded known mappings
    if outcode in OUTCODE_ID_STATIC:
        return OUTCODE_ID_STATIC[outcode]
    if outcode in OUTCODE_ID_CACHE:
        return OUTCODE_ID_CACHE[outcode]

    # 1) Try typeahead/suggest endpoints that often return identifiers
    suggest_urls = [
        f"https://www.rightmove.co.uk/api/suggest/v1/locations?query={outcode}&channel=BUY",
        f"https://www.rightmove.co.uk/typeAhead/uknostreet/BUY/0/{outcode}.json",
    ]
    for su in suggest_urls:
        try:
            r = SESSION.get(su, timeout=20)
            if r.ok:
                data = r.json()
                li = _search_json_for_outcode_identifier(data)
                if li:
                    OUTCODE_ID_CACHE[outcode] = li
                    return li
        except Exception:
            pass

    # 2) Try a minimal search and inspect redirect URL or HTML content
    params = {
        "searchLocation": outcode,
        "useLocationIdentifier": "true",
        "channel": "BUY",
        "transactionType": "BUY",
        "index": "0",
    }
    url = "https://www.rightmove.co.uk/property-for-sale/find.html?" + urlencode(params, safe="^")
    try:
        r = SESSION.get(url, timeout=30, allow_redirects=True)
        q = parse_qs(urlparse(r.url).query)
        li_vals = q.get("locationIdentifier")
        if li_vals and li_vals[0].startswith("OUTCODE^"):
            OUTCODE_ID_CACHE[outcode] = li_vals[0]
            return li_vals[0]
        m = re.search(r"OUTCODE\^(\d+)", r.text)
        if m:
            li = f"OUTCODE^{m.group(1)}"
            OUTCODE_ID_CACHE[outcode] = li
            return li
    except requests.RequestException:
        return None
    return None


def build_outcode_start_url(outcode: str) -> str:
    # Use the simpler path form that works without numeric OUTCODE id
    return f"https://www.rightmove.co.uk/property-for-sale/{outcode.upper().strip()}.html"

def extract_property_links_via_api(start_url: str, idx: int) -> list[str]:
    parsed = urlparse(start_url)
    q = parse_qs(parsed.query)
    li_vals = q.get("locationIdentifier")
    if not li_vals:
        return []
    li = li_vals[0]
    api_params = {
        "locationIdentifier": li,
        "channel": "BUY",
        "index": str(idx),
        "radius": q.get("radius", ["0.0"])[0],
        "sortType": q.get("sortType", ["2"])[0],
        "viewType": q.get("viewType", ["LIST"])[0],
        "includeSSTC": "true",
        "maxDaysSinceAdded": q.get("maxDaysSinceAdded", ["14"])[0],
    }
    url = "https://www.rightmove.co.uk/api/_search?" + urlencode(api_params, safe="^")
    try:
        r = SESSION.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        props = data.get("properties") or []
        links = []
        for p in props:
            pid = p.get("id") or p.get("propertyId")
            if pid:
                links.append(f"https://www.rightmove.co.uk/properties/{pid}")
        return links
    except Exception:
        return []


def paginate_and_scrape_outcode(start_url: str, max_pages: int = 2) -> pd.DataFrame:
    all_rows = []
    parsed = urlparse(start_url)
    def with_index(idx: int) -> str:
        if "index=" in start_url:
            return re.sub(r"index=\d+", f"index={idx}", start_url)
        sep = "&" if parsed.query else "?"
        return f"{start_url}{sep}index={idx}"
    for page_no in range(max_pages):
        idx = page_no * 24
        url = with_index(idx)
        print(f"Listing page {page_no+1}/{max_pages}: index={idx}")
        links = extract_property_links(url)
        if not links:
            links = extract_property_links_via_api(start_url, idx)
        if not links:
            print("No property links found, stopping.")
            break
        print(f"Found {len(links)} property links")
        for link in links:
            try:
                row = extract_detail(link) if 'extract_detail' in globals() else {"url": link, "property_id": re.search(r"/properties/(\d+)", link).group(1) if re.search(r"/properties/(\d+)", link) else None, "address": None}
                all_rows.append(row)
            except Exception:
                continue
    return pd.DataFrame(all_rows)

# 4) Test scrape for only these outcodes first
TEST_OUTCODES = ["B93", "B92", "B94"]
MAX_PAGES_TO_SCRAPE = 2  # keep small while testing

all_frames = []
for oc in TEST_OUTCODES:
    start_url = build_outcode_start_url(oc)
    print(f"\n=== Scraping OUTCODE {oc} ===")
    print(f"Start URL: {start_url}")
    if not start_url:
        continue
    df = paginate_and_scrape_outcode(start_url, max_pages=MAX_PAGES_TO_SCRAPE)
    if df.empty:
        continue
    df["outcode"] = oc
    df["district"] = df["address"].apply(extract_outward_postcode) if "address" in df.columns else oc
    all_frames.append(df)

if all_frames:
    results_df = pd.concat(all_frames, ignore_index=True)
    # Insert outcode after property_id
    cols = list(results_df.columns)
    if "outcode" in cols and "property_id" in cols:
        cols.remove("outcode")
        insert_at = cols.index("property_id") + 1
        cols.insert(insert_at, "outcode")
        results_df = results_df[cols]
    print(f"Collected {len(results_df)} properties across {len(all_frames)} outcodes")
    out_path = "rightmove_outcodes_test.csv"
    results_df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")
else:
    print("No data scraped for test outcodes.")



In [ ]:
# Full outcode scrape (all outcodes) -> rm_wm_property.csv
import pandas as pd
import re

# Build list of outcodes to scrape
try:
    outcodes_to_scrape = (
        oc_df["outcode"].dropna().astype(str).str.upper().str.replace(r"\s+", "", regex=True).unique().tolist()
    )
except Exception:
    oc_df2 = pd.read_csv("outward_postcodes.csv")
    col_candidates = ["OutwardPostcode", "Outward Postcode", "outward_postcode", "outcode", "district"]
    outcode_col = None
    for c in col_candidates:
        if c in oc_df2.columns:
            outcode_col = c
            break
    if outcode_col is None:
        raise ValueError("Could not find an outcode column in outward_postcodes.csv.")
    outcodes_to_scrape = (
        oc_df2[outcode_col].dropna().astype(str).str.upper().str.replace(r"\s+", "", regex=True).unique().tolist()
    )

print(f"Total outcodes to scrape: {len(outcodes_to_scrape)}")

MAX_PAGES_PER_OUTCODE = 200
all_frames = []

for i, oc in enumerate(outcodes_to_scrape, 1):
    start_url = build_outcode_start_url(oc)
    print(f"\n[{i}/{len(outcodes_to_scrape)}] OUTCODE {oc}")
    print(f"Start URL: {start_url}")
    df = paginate_and_scrape_outcode(start_url, max_pages=MAX_PAGES_PER_OUTCODE)
    if df.empty:
        continue
    df["outcode"] = oc
    df["district"] = df["address"].apply(extract_outward_postcode) if "address" in df.columns else oc
    all_frames.append(df)

if all_frames:
    results_df = pd.concat(all_frames, ignore_index=True)
    if "property_id" in results_df.columns:
        results_df = results_df.drop_duplicates(subset=["property_id"])  # dedupe
    # Insert outcode after property_id
    cols = list(results_df.columns)
    if "outcode" in cols and "property_id" in cols:
        cols.remove("outcode")
        insert_at = cols.index("property_id") + 1
        cols.insert(insert_at, "outcode")
        results_df = results_df[cols]
    out_path = "rm_wm_property.csv"
    results_df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path} ({len(results_df)} rows)")
else:
    print("No data scraped.")



In [ ]:
# Full outcode scrape (self-contained) -> rm_wm_property.csv
import re
import json
import time
from typing import List, Dict, Optional
from urllib.parse import urlparse, parse_qs

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------- Config ----------
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,application/json;q=0.8,*/*;q=0.7",
    "Accept-Language": "en-GB,en;q=0.9",
    "Connection": "keep-alive",
}
SESSION = requests.Session()
SESSION.headers.update(REQUEST_HEADERS)

PROPERTY_LINK_SELECTORS = [
    "a.propertyCard-link",
    "a.propertyCard-detailsLink",
    "a[data-test='property-card-link']",
]

# ---------- Helpers ----------

def build_outcode_url(outcode: str) -> str:
    return f"https://www.rightmove.co.uk/property-for-sale/{outcode.upper().strip()}.html"


def add_index(url: str, idx: int) -> str:
    return url + ("&" if urlparse(url).query else "?") + f"index={idx}"


def get_html(url: str) -> Optional[str]:
    try:
        r = SESSION.get(url, timeout=30)
        r.raise_for_status()
        return r.text
    except requests.RequestException:
        return None


def get_soup(url: str) -> Optional[BeautifulSoup]:
    html = get_html(url)
    if html is None:
        return None
    return BeautifulSoup(html, "html.parser")


def extract_property_links_html(url: str) -> List[str]:
    soup = get_soup(url)
    if soup is None:
        return []
    links: List[str] = []
    for selector in PROPERTY_LINK_SELECTORS:
        for a in soup.select(selector):
            href = a.get("href")
            if not href:
                continue
            if href.startswith("/"):
                href = "https://www.rightmove.co.uk" + href
            if "/properties/" in href:
                links.append(href.split("#")[0])
    return sorted(set(links))


def find_outcode_location_identifier_from_html(html: str) -> Optional[str]:
    if not html:
        return None
    m = re.search(r"OUTCODE\^(\d+)", html)
    return f"OUTCODE^{m.group(1)}" if m else None


def extract_property_links_api(li: str, idx: int) -> List[str]:
    params = {
        "locationIdentifier": li,
        "channel": "BUY",
        "index": str(idx),
        "includeSSTC": "true",
        "sortType": "2",
    }
    url = "https://www.rightmove.co.uk/api/_search" \
        + ("?" + "&".join([f"{k}={v}" for k, v in params.items()]))
    try:
        r = SESSION.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        props = data.get("properties") or []
        links: List[str] = []
        for p in props:
            pid = p.get("id") or p.get("propertyId")
            if pid:
                links.append(f"https://www.rightmove.co.uk/properties/{pid}")
        return links
    except Exception:
        return []


INT_PATTERN = re.compile(r"(\d+)")
SQM_PATTERN = re.compile(r"([0-9,.]+)\s*(sq\s*m|sqm|square\s*metres?)", re.I)
SQFT_PATTERN = re.compile(r"([0-9,.]+)\s*(sq\s*ft|sqft|square\s*feet)\b", re.I)


def first_int(text: Optional[str]) -> Optional[int]:
    if not text:
        return None
    m = INT_PATTERN.search(text)
    return int(m.group(1)) if m else None


def extract_outward_postcode(address: str) -> Optional[str]:
    if not isinstance(address, str) or not address.strip():
        return None
    text = address.upper().strip().rstrip(",.;")
    m = re.search(r"([A-Z]{1,2}\d{1,2}[A-Z]?)\s*\d[ABD-HJLNP-UW-Z]{2}$", text)
    return m.group(1) if m else None


def extract_detail(url: str) -> Dict:
    result: Dict = {
        "url": url,
        "property_id": None,
        "address": None,
        "price": None,
        "property_type": None,
        "bedrooms": None,
        "bathrooms": None,
        "size_sq_m": None,
        "size_sq_ft": None,
        "tenure": None,
        "council_tax_band": None,
        "parking": None,
        "garden": None,
        "key_features": None,
    }
    try:
        m = re.search(r"/properties/(\d+)", url)
        if m:
            result["property_id"] = m.group(1)
        r = SESSION.get(url, timeout=30)
        r.raise_for_status()
        html = r.text
        soup = BeautifulSoup(html, "html.parser")
        # price
        price_el = soup.select_one("[data-testid='price']") or soup.select_one(".property-header-price")
        if price_el:
            result["price"] = price_el.get_text(" ", strip=True)
        # address
        addr_el = soup.select_one("[data-testid='address']") or soup.select_one(".address, [itemprop='address']")
        if addr_el:
            result["address"] = addr_el.get_text(" ", strip=True)
        # title-derived fields
        title_el = soup.select_one("[data-testid='title']") or soup.select_one("h1")
        if title_el:
            title_text = title_el.get_text(" ", strip=True)
            if result["bedrooms"] is None:
                bm = re.search(r"(\d+)\s*bed", title_text, re.I)
                if bm:
                    result["bedrooms"] = int(bm.group(1))
            if result["property_type"] is None:
                tm = re.search(r"\b(Detached|Semi[- ]?Detached|End of Terrace|Terraced|Flat|Apartment|Bungalow|Cottage|Townhouse)\b", title_text, re.I)
                if tm:
                    result["property_type"] = tm.group(1).replace("-", " ").title()
        # features
        feats = []
        for ul_sel in ["ul.key-features", "ul.property-features", "ul[data-test='key-features']", "article ul"]:
            for ul in soup.select(ul_sel):
                for li in ul.select("li"):
                    t = li.get_text(" ", strip=True)
                    if t:
                        feats.append(t)
        if feats:
            seen = set(); ordered = []
            for f in feats:
                if f not in seen:
                    seen.add(f); ordered.append(f)
            result["key_features"] = "; ".join(ordered)
        # tenure / council tax via dt/dd pairs
        for dt in soup.select("dt"):
            key = dt.get_text(strip=True).upper()
            dd = dt.find_next_sibling("dd")
            if not dd:
                continue
            val = dd.get_text(" ", strip=True)
            if "TENURE" in key and not result["tenure"]:
                result["tenure"] = val
            if "COUNCIL TAX" in key and not result["council_tax_band"]:
                m2 = re.search(r"Band\s*:?\s*([A-H])", val, re.I)
                result["council_tax_band"] = (m2.group(1).upper() if m2 else val)
        return result
    except Exception:
        return result


def paginate_outcode(outcode: str, max_pages: int = 200, delay_s: float = 0.6) -> pd.DataFrame:
    start_url = build_outcode_url(outcode)
    print(f"Start URL: {start_url}")
    html0 = get_html(start_url) or ""
    li = find_outcode_location_identifier_from_html(html0)

    all_rows: List[Dict] = []
    for page_no in range(max_pages):
        idx = page_no * 24
        url = add_index(start_url, idx)
        print(f"  Listing page {page_no+1}/{max_pages}: index={idx}")
        links = extract_property_links_html(url)
        if not links and li:
            links = extract_property_links_api(li, idx)
        if not links:
            print("  No property links found, stopping.")
            break
        print(f"  Found {len(links)} property links")
        for link in links:
            row = extract_detail(link)
            all_rows.append(row)
            time.sleep(delay_s)
        time.sleep(0.8)
    return pd.DataFrame(all_rows)


def load_outcodes() -> List[str]:
    # Prefer existing oc_df if present in the notebook
    if "oc_df" in globals() and isinstance(globals()["oc_df"], pd.DataFrame):
        series = globals()["oc_df"].get("outcode")
        if series is not None:
            outcodes = (series.dropna().astype(str).str.upper().str.replace(r"\s+", "", regex=True).unique().tolist())
            if outcodes:
                return outcodes
    # Fallback to outward_postcodes.csv
    df = pd.read_csv("outward_postcodes.csv")
    for c in ["OutwardPostcode", "Outward Postcode", "outward_postcode", "outcode", "district"]:
        if c in df.columns:
            series = df[c]
            return (series.dropna().astype(str).str.upper().str.replace(r"\s+", "", regex=True).unique().tolist())
    raise ValueError("Could not find an outcode column in outward_postcodes.csv")


# ---------- Run full scrape ----------
OUTCODES = load_outcodes()
print(f"Total outcodes to scrape: {len(OUTCODES)}")

all_frames: List[pd.DataFrame] = []
for i, oc in enumerate(OUTCODES, 1):
    print(f"\n[{i}/{len(OUTCODES)}] OUTCODE {oc}")
    df = paginate_outcode(oc, max_pages=200)
    if df.empty:
        continue
    df["outcode"] = oc
    # Derive outward from address if present
    if "address" in df.columns:
        df["district"] = df["address"].apply(extract_outward_postcode)
    else:
        df["district"] = oc
    all_frames.append(df)

if all_frames:
    results_df = pd.concat(all_frames, ignore_index=True)
    if "property_id" in results_df.columns:
        results_df = results_df.drop_duplicates(subset=["property_id"])  # dedupe
    # Insert outcode after property_id if both exist
    cols = list(results_df.columns)
    if "outcode" in cols and "property_id" in cols:
        cols.remove("outcode")
        insert_at = cols.index("property_id") + 1
        cols.insert(insert_at, "outcode")
        results_df = results_df[cols]
    out_path = "rm_wm_property.csv"
    results_df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path} ({len(results_df)} rows)")
else:
    print("No data scraped.")

